# Step 3.2: features on other timeframes and other instruments, with their warm-up

This notebook is the check of step 3.2. `ff.features.compute` gives a window's signal bars with a column for each feature, which the feature runtime evaluates with what the feature declares beside its bars: a feature of 15-minute bars, whose value a 5-minute row sees only once that 15-minute bar has closed; another product's bars, joined at each close with their staleness, the time since that product's last bar closed; and the features a feature reads, each evaluated before it. Each feature's warm-up, the bars before its values mean what they say, is read from the sessions before the window, so the window's first rows have their values.

Pick the `.venv` kernel and run all cells. It reads the store in `FEFAT_DATA_DIR` and writes nothing.

## A 15-minute RSI on 5-minute bars

`rsi_15m` declares the 15-minute timeframe and reads the example RSI, so the runtime works RSI(14) out on 15-minute bars and gives each 5-minute row the value of the last 15-minute bar closed by that row's close. The product is MES when your store holds its 1-minute bars, else the product with the most contracts; the week is the one to the last trade date of its data, and the day the last one whose bars reach 09:45 in Chicago.

Expected: the product and the day, then its 5-minute rows from 09:20 to 09:45 Chicago time, with the close in points, the 5-minute RSI, and `rsi_15m`, which keeps one value at 09:20 and 09:25, changes at 09:30, when the 15-minute bar from 09:15 closes, keeps it to 09:40, and changes again at 09:45; then `changes only at 15-minute closes: True` over the whole week.

In [ ]:
from datetime import date, time, timedelta

import polars as pl
from IPython.display import display

import fefat as ff
from fefat.features import InstrumentInput, TimeframeInput
from fefat.features.examples import rsi


@ff.feature(inputs=[rsi, TimeframeInput(timeframe="15m")], outputs={"rsi_15m": "ratio"})
def rsi_15m(bars: pl.LazyFrame) -> pl.Expr:
    return pl.col("rsi")


CHICAGO = "America/Chicago"
FIFTEEN_MINUTES = 15 * 60 * 10**9
MINUTE = 60 * 10**9
listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
roots = set(listed.get_column("root"))
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, LAST = chosen["root"], chosen["last_trade_date"]
local_close = (
    pl.from_epoch("ts_close", time_unit="ns")
    .dt.replace_time_zone("UTC")
    .dt.convert_time_zone(CHICAGO)
    .alias("close_chicago")
)
slots = {"rsi": rsi(n=14), "rsi_15m": rsi_15m()}
week = ff.features.compute(slots, ROOT, "5m", LAST - timedelta(days=7), LAST).collect()
DAY = week.filter(local_close.dt.time() == time(9, 45)).get_column("trade_date").max()
print(f"{ROOT} on {DAY}")
around = week.filter(
    pl.col("trade_date") == DAY, local_close.dt.time().is_between(time(9, 20), time(9, 45))
)
display(around.select(local_close, (pl.col("close") / 1e9).alias("close_points"), "rsi", "rsi_15m"))
after_the_first = pl.int_range(pl.len()) > 0
changed = week.filter(pl.col("rsi_15m").ne_missing(pl.col("rsi_15m").shift(1)), after_the_first)
at_closes = (changed.get_column("ts_close") % FIFTEEN_MINUTES == 0).all()
print("changes only at 15-minute closes:", at_closes)

## Another product's bars

Each feature below reads one other product's bars at the same timeframe, joined at each 5-minute close, and gives their staleness in nanoseconds, printed in minutes. MNQ keeps MES's calendar and grid, and MCL keeps its hours on ordinary days; the fixtures' bitcoin trades on UTC days, so ES's rows read nothing before its Saturday and its last Saturday bar after it.

Expected: for each other product your store holds among MNQ, MCL, and BTCUSDT-PERP, the week's rows that read a bar of the same close, those that read an older bar, with the oldest in minutes, and those before its first bar; on your store, MNQ and MCL read a bar of the same close on every row of the week.

In [ ]:
@ff.feature(inputs=[InstrumentInput(root="MNQ")], outputs={"staleness": "other"})
def mnq_staleness(bars: pl.LazyFrame) -> pl.Expr:
    return pl.col("staleness_MNQ")


@ff.feature(inputs=[InstrumentInput(root="MCL")], outputs={"staleness": "other"})
def mcl_staleness(bars: pl.LazyFrame) -> pl.Expr:
    return pl.col("staleness_MCL")


@ff.feature(inputs=[InstrumentInput(root="BTCUSDT-PERP")], outputs={"staleness": "other"})
def btc_staleness(bars: pl.LazyFrame) -> pl.Expr:
    return pl.col("staleness_BTCUSDT-PERP")


others = {"MNQ": mnq_staleness(), "MCL": mcl_staleness(), "BTCUSDT-PERP": btc_staleness()}
held = {other: spec for other, spec in others.items() if other in roots and other != ROOT}
ages = ff.features.compute(held, ROOT, "5m", LAST - timedelta(days=7), LAST).collect()
for other in held:
    staleness = ages.get_column(other)
    read = staleness.drop_nulls()
    print(
        f"{other}: {(read == 0).sum()} rows read a bar of the same close, {(read > 0).sum()} "
        f"an older bar, the oldest {read.max() / MINUTE:,.0f} minutes old, and "
        f"{staleness.null_count()} rows come before its first bar"
    )

## Staleness across a halt

On Labor Day, 2026-09-07, CME's equity index products halted at 12:00 Chicago and its energy products at 13:30, both trading for the trade date of 09-08. A feature of MCL reading MES sees MES's last bar before its halt on each MCL row after it.

Expected: where your store holds MCL and MES, MCL's 5-minute rows of 2026-09-07 from 11:55 to 13:30 Chicago with MES's staleness in minutes, 0 to 12:00, then 5, 10, and so on to 90 at 13:30, and the same minutes on one line; on a store without them, a line saying the part is skipped.

In [ ]:
@ff.feature(inputs=[InstrumentInput(root="MES")], outputs={"staleness": "other"})
def mes_staleness(bars: pl.LazyFrame) -> pl.Expr:
    return pl.col("staleness_MES")


if {"MCL", "MES"} <= roots:
    halted = ff.features.compute(
        {"MES": mes_staleness()}, "MCL", "5m", "2026-09-08", "2026-09-08"
    ).collect()
    labor_day = halted.filter(
        local_close.dt.date() == date(2026, 9, 7),
        local_close.dt.time().is_between(time(11, 55), time(13, 30)),
    )
    display(labor_day.select(local_close, (pl.col("MES") / MINUTE).alias("mes_staleness_minutes")))
    print("MES staleness in minutes:", (labor_day.get_column("MES") // MINUTE).to_list())
else:
    print("The store holds no MCL and MES: this part is skipped.")

## The warm-up read from before the window

RSI(14) needs 140 bars before its values mean what they say. A window of the one day reads them from the sessions before it, so its first 5-minute row has its RSI.

Expected: the day's first row with its RSI, a number, and `first row has its RSI: True`.

In [ ]:
one_day = ff.features.compute({"rsi": rsi(n=14)}, ROOT, "5m", DAY, DAY).collect()
display(one_day.head(1).select(local_close, "trade_date", "rsi"))
print("first row has its RSI:", one_day.item(0, "rsi") is not None)